# OpenNeuro Brain EDA

This notebook summarizes BIDS metadata, task timing, protocol groupings, and any locally downloaded NIfTI headers.

In [ ]:
import csv
import json
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

import nibabel as nib

from thesis_neuro.paths import data_root, output_root


In [ ]:
# Inputs resolve under THESIS_NEURO_DATA_ROOT and outputs under THESIS_NEURO_OUTPUT_ROOT.
DATA_ROOT = data_root()
OUTPUT_ROOT = output_root()
REPORTS_DIR = OUTPUT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DS002345_DIR = DATA_ROOT / "openneuro" / "ds002345"
DS002345_TASKS = {'black', 'bronx', 'forgot', 'piemanpni', 'shapesphysical', 'shapessocial'}

DS002322_DIR = DATA_ROOT / "openneuro" / "ds002322"

ANNEX_SIZE_RE = re.compile(r'MD5E-s(\d+)--')
PROTOCOL_FIELDS = [
    'TaskName', 'Manufacturer', 'ManufacturersModelName', 'MagneticFieldStrength',
    'RepetitionTime', 'EchoTime', 'FlipAngle', 'SliceThickness', 'SpacingBetweenSlices',
    'AcquisitionMatrixPE', 'ReconMatrixPE', 'PixelBandwidth', 'MultibandAccelerationFactor',
    'ParallelReductionFactorInPlane', 'PhaseEncodingDirection'
]

In [ ]:
def load_json(path):
    with path.open() as f:
        return json.load(f)


def load_tsv(path):
    with path.open(newline='') as f:
        return list(csv.DictReader(f, delimiter='\t'))


def bids_entity(path, prefix):
    for part in path.name.split('_'):
        if part.startswith(prefix):
            return part[len(prefix):]
    return None


def annex_size_bytes(path):
    if not path.is_symlink():
        return path.stat().st_size if path.exists() else None
    target = os.readlink(path)
    match = ANNEX_SIZE_RE.search(target)
    return int(match.group(1)) if match else None


def has_local_content(path):
    try:
        return path.exists() and path.resolve(strict=True).exists()
    except FileNotFoundError:
        return False


def local_header_summary(path):
    img = nib.load(str(path))
    zooms = img.header.get_zooms()
    return {
        'path': str(path),
        'shape': list(img.shape),
        'voxel_size_mm': [float(v) for v in zooms[:3]],
        'tr_seconds': float(zooms[3]) if len(zooms) > 3 else None,
        'dtype': str(img.header.get_data_dtype()),
    }


def summarize_events(event_paths):
    by_task = {}
    for path in event_paths:
        task = bids_entity(path, 'task-')
        rows = load_tsv(path)
        trial_types = Counter()
        stim_files = Counter()
        max_end = 0.0
        for row in rows:
            try:
                duration = float(row['duration'])
                onset = float(row['onset'])
            except (TypeError, ValueError):
                continue
            trial_types[row.get('trial_type', 'n/a')] += duration
            stim = row.get('stim_file')
            if stim:
                stim_files[stim] += 1
            max_end = max(max_end, onset + duration)
        task_summary = by_task.setdefault(task, {
            'event_files': 0,
            'unique_stim_files': set(),
            'trial_type_duration_seconds': Counter(),
            'scan_end_times_seconds': [],
        })
        task_summary['event_files'] += 1
        task_summary['unique_stim_files'].update(stim_files)
        task_summary['trial_type_duration_seconds'].update(trial_types)
        task_summary['scan_end_times_seconds'].append(max_end)

    for task, summary in by_task.items():
        summary['unique_stim_files'] = sorted(summary['unique_stim_files'])
        summary['trial_type_duration_seconds'] = dict(summary['trial_type_duration_seconds'])
        ends = summary['scan_end_times_seconds']
        summary['scan_end_times_seconds'] = {
            'min': min(ends) if ends else None,
            'max': max(ends) if ends else None,
            'unique': sorted({round(v, 3) for v in ends}),
        }
    return by_task


def summarize_protocols(json_paths):
    groups = defaultdict(lambda: {'count': 0, 'example_path': None, 'fields': None})
    for path in json_paths:
        data = load_json(path)
        key = tuple(data.get(field) for field in PROTOCOL_FIELDS)
        groups[key]['count'] += 1
        if groups[key]['example_path'] is None:
            groups[key]['example_path'] = str(path)
            groups[key]['fields'] = {field: data.get(field) for field in PROTOCOL_FIELDS}
    return list(groups.values())


def summarize_downloaded_headers(paths):
    results = []
    for path in paths:
        if has_local_content(path):
            results.append(local_header_summary(path))
    return results


def build_report(dataset_dir, tasks=None):
    tasks = set(tasks or [])
    bold_paths = []
    bold_json_paths = []
    event_paths = []
    for path in dataset_dir.rglob('*_bold.nii.gz'):
        task = bids_entity(path, 'task-')
        if tasks and task not in tasks:
            continue
        bold_paths.append(path)
    for path in dataset_dir.rglob('*_bold.json'):
        task = bids_entity(path, 'task-')
        if tasks and task not in tasks:
            continue
        bold_json_paths.append(path)
    for path in dataset_dir.rglob('*_events.tsv'):
        task = bids_entity(path, 'task-')
        if tasks and task not in tasks:
            continue
        event_paths.append(path)

    t1_paths = sorted(dataset_dir.rglob('*_T1w.nii.gz'))
    dataset_description = load_json(dataset_dir / 'dataset_description.json')
    participants_tsv = dataset_dir / 'participants.tsv'
    participants_rows = load_tsv(participants_tsv) if participants_tsv.exists() else []

    bold_tasks = Counter(bids_entity(path, 'task-') for path in bold_paths)
    bold_subjects = sorted({bids_entity(path, 'sub-') for path in bold_paths if bids_entity(path, 'sub-')})
    total_bold_bytes = sum(size for path in bold_paths if (size := annex_size_bytes(path)) is not None)

    return {
        'dataset': dataset_description.get('Name'),
        'dataset_id': dataset_dir.name,
        'selected_tasks': sorted(tasks) if tasks else 'ALL',
        'dataset_doi': dataset_description.get('DatasetDOI'),
        'bids_version': dataset_description.get('BIDSVersion'),
        'participants_tsv_rows': len(participants_rows),
        'bold_file_count': len(bold_paths),
        'bold_subject_count': len(bold_subjects),
        'bold_task_counts': dict(bold_tasks),
        'estimated_total_bold_compressed_size_bytes': total_bold_bytes,
        'estimated_total_bold_compressed_size_gb': round(total_bold_bytes / (1024 ** 3), 2),
        'event_summary_by_task': summarize_events(sorted(event_paths)),
        'protocol_groups': summarize_protocols(sorted(bold_json_paths)),
        'downloaded_bold_headers': summarize_downloaded_headers(sorted(bold_paths)),
        'downloaded_t1w_headers': summarize_downloaded_headers(sorted(t1_paths)),
    }

In [ ]:
ds002345_report = build_report(DS002345_DIR, tasks=DS002345_TASKS)
ds002345_path = REPORTS_DIR / 'ds002345_targeted_eda.json'
ds002345_path.write_text(json.dumps(ds002345_report, indent=2))
ds002345_path

In [ ]:
ds002345_report['dataset'], ds002345_report['bold_task_counts'], ds002345_report['estimated_total_bold_compressed_size_gb']

In [ ]:
ds002322_report = build_report(DS002322_DIR)
ds002322_path = REPORTS_DIR / 'ds002322_eda.json'
ds002322_path.write_text(json.dumps(ds002322_report, indent=2))
ds002322_path

In [ ]:
ds002322_report['dataset'], ds002322_report['bold_task_counts'], ds002322_report['estimated_total_bold_compressed_size_gb']